# Using Common Crawl data via Hugging Face's S3-compatible gateway

The Common Crawl data stored on [the Hugging Face bucket](https://huggingface.co/buckets/commoncrawl/commoncrawl) should usually be accessed with HF-native tools and APIs (see other notebooks). However, Hugging Face also provides an [S3-compatible gateway for its storage buckets](https://huggingface.co/docs/hub/storage-buckets-s3), i.e., if you already use S3, this is a way to reuse your existing tooling.

## Generating S3 Credentials

Follow the instructions in the [official documentation](https://huggingface.co/docs/hub/storage-buckets-s3#generating-s3-credentials) to generate S3 credentials for your Hugging Face account. The credentials will be similar to those below and should be added to your `~/.aws/credentials` file together with an updated AWS profile:

### Example: AWS CLI profile

Add a profile to `~/.aws/config`:

```
[profile hf]
region = us-east-1
endpoint_url = https://s3.hf.co/commoncrawl
s3 =
    addressing_style = path
    multipart_threshold = 2GB
    multipart_chunksize = 2GB
request_checksum_calculation = when_required
response_checksum_validation = when_required
```

Note: The official documentation uses a `<namespace>` placeholder. The placeholder needs to be replaced with `commoncrawl`.

Add the matching credentials for the profile to `~/.aws/credentials`:

```
[hf]
aws_access_key_id = HFAK...
aws_secret_access_key = ...
```

## Accessing CC HF bucket with AWS CLI

Now you can use the AWS CLI to access the Common Crawl HF bucket by using the `--profile hf` argument: 

In [ ]:
# what crawl data is on HF?
!aws --profile hf s3 ls s3://commoncrawl/crawl-data/

                           PRE CC-MAIN-2026-17/
                           PRE CC-MAIN-2026-21/
                           PRE CC-MAIN-2026-25/


In [5]:
# how large is the index data ?
!aws --profile hf s3 ls s3://commoncrawl/cc-index/ --recursive --human-readable --summarize | tail -n 3


Total Objects: 2700
   Total Size: 590.7 GiB


## DuckDB via the S3 gateway

To use DuckDB for index queries, you need to set up the S3 credentials as follows:

```sql
-- 1. Create the S3-compatible secret
CREATE OR REPLACE SECRET secret (
    TYPE s3,
    PROVIDER config,
    KEY_ID 'xxx',
    SECRET 'yyy',
    REGION 'us-east-1',
    ENDPOINT 's3.hf.co/commoncrawl',
    URL_STYLE 'path'
);
```

And then you can use the same queries as you would with AWS S3 but via HF:

```sql
SELECT * FROM 's3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-17/subset=crawldiagnostics/part-00000-ee7620e8-27a7-4c74-846b-a92b039dae92.c000.gz.parquet' 
LIMIT 5;
```

```
┌────────────────────────────────┬──────────────────────┬──────────────────────┬───┬──────────────────┬─────────────────┬──────────────────┐
│          url_surtkey           │         url          │    url_host_name     │ … │   warc_segment   │      crawl      │      subset      │
│            varchar             │       varchar        │       varchar        │ … │     varchar      │     varchar     │     varchar      │
├────────────────────────────────┼──────────────────────┼──────────────────────┼───┼──────────────────┼─────────────────┼──────────────────┤
│ ru,yandex,market-delivery)/zo… │ https://market-deli… │ market-delivery.yan… │ … │ 1775805909101.46 │ CC-MAIN-2026-17 │ crawldiagnostics │
│ ru,yandex,market-delivery)/zo… │ https://market-deli… │ market-delivery.yan… │ … │ 1775806209748.93 │ CC-MAIN-2026-17 │ crawldiagnostics │
│ ru,yandex,market-delivery)/zo… │ https://market-deli… │ market-delivery.yan… │ … │ 1775805908721.20 │ CC-MAIN-2026-17 │ crawldiagnostics │
│ ru,yandex,market-delivery)/zo… │ https://market-deli… │ market-delivery.yan… │ … │ 1775805908780.58 │ CC-MAIN-2026-17 │ crawldiagnostics │
│ ru,yandex,market-delivery)/zo… │ https://market-deli… │ market-delivery.yan… │ … │ 1775805908654.63 │ CC-MAIN-2026-17 │ crawldiagnostics │
└────────────────────────────────┴──────────────────────┴──────────────────────┴───┴──────────────────┴─────────────────┴──────────────────┘
```